# WFJSSP-GA Demo Workflow

Dieses Notebook ist auf einen demo-orientierten Ablauf ausgelegt:
- Batch-Auswertung ueber mehrere oder alle FJSSP-W-Instanzen
- mehrere unabhaengige Runs pro Instanz
- getrennte finale Bewertung fuer Szenario 1 oder 2
- Export tabellarischer Resultate

Fuer Szenario 2 werden die Unsicherheitsparameter in dieser Demo standardmaessig mit `create_uncertainty_vector(...)` erzeugt.

In [2]:
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from solver.GA.wfjssp_ga import build_ga_from_worker_encoding
from util.benchmark_parser import WorkerBenchmarkParser
from util.evaluation import makespan, translate, workload_balance
from util.graph import run_n_simulations
from util.uncertainty import create_uncertainty_vector

plt.style.use("ggplot")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)


In [ ]:
INSTANCES_DIR = Path("instances/fjssp-w")
RESULTS_DIR = Path("results/wfjssp_ga_competition")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO = 2  # 1 = deterministisch, 2 = Unsicherheit
STRICT_COMPETITION_MODE = False

# Optional: Datei mit offiziellen Unsicherheitsparametern.
# Erwartetes Format: {"instance_name.fjs": [[alpha, beta, offset], ...], ...}
OFFICIAL_UNCERTAINTY_FILE = None

# Standardverhalten der Demo: Parameter werden lokal erzeugt.
ALLOW_DEMO_UNCERTAINTY_FALLBACK = True
DEMO_UNCERTAINTY_SEED = 123
DEMO_UNCERTAINTY_FACTOR = 10.0
DEMO_UNCERTAINTY_OFFSET = 1.0
UNCERTAINTY_SOURCE = "worker"

# Competition-relevante Wiederholungen
N_INDEPENDENT_RUNS = 10
FINAL_EVAL_SIMULATIONS = 5

# Zum schnellen Testen kann die Auswahl reduziert werden.
SELECTED_INSTANCES = sorted(p.name for p in INSTANCES_DIR.glob("*.fjs"))
# SELECTED_INSTANCES = ["1_Brandimarte_7_workers.fjs"]

RUN_SEEDS = [1000 + i for i in range(N_INDEPENDENT_RUNS)]

GA_CONFIG = {
    "population_size": 100,
    "offspring_amount": 400,
    "elitism_rate": 0.1,
    "restart_generations": 100,
}

RUN_CONFIG = {
    "max_generations": 250,
    "time_limit_s": 20,
    "keep_multiple": False,
    "do_restart": True,
}

pd.Series(
    {
        "scenario": SCENARIO,
        "strict_competition_mode": STRICT_COMPETITION_MODE,
        "instances": len(SELECTED_INSTANCES),
        "independent_runs": N_INDEPENDENT_RUNS,
        "final_eval_simulations": FINAL_EVAL_SIMULATIONS if SCENARIO == 2 else 0,
        "results_dir": str(RESULTS_DIR),
    }
)

scenario                                               2
strict_competition_mode                            False
instances                                             30
independent_runs                                      10
final_eval_simulations                                50
results_dir                results/wfjssp_ga_competition
dtype: object

In [4]:
def load_official_uncertainty_map(path):
    if path is None:
        return None
    path = Path(path)
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def get_uncertainty_parameters(instance_name, encoding, uncertainty_map):
    if SCENARIO != 2:
        return None, "disabled"

    if uncertainty_map is not None:
        if instance_name not in uncertainty_map:
            raise KeyError(f"No uncertainty parameters found for {instance_name}")
        return uncertainty_map[instance_name], "official"

    if STRICT_COMPETITION_MODE:
        raise ValueError(
            "Scenario 2 in strict competition mode requires OFFICIAL_UNCERTAINTY_FILE with provided parameters."
        )

    if not ALLOW_DEMO_UNCERTAINTY_FALLBACK:
        return None, "missing"

    random.seed(DEMO_UNCERTAINTY_SEED)
    resource_count = encoding.durations().shape[2] if UNCERTAINTY_SOURCE == "worker" else encoding.durations().shape[1]
    params = create_uncertainty_vector(
        resource_count,
        factor=DEMO_UNCERTAINTY_FACTOR,
        offset=DEMO_UNCERTAINTY_OFFSET,
    )
    return params, "demo_fallback"


def solve_instance_with_ga(encoding, seed, uncertainty_parameters=None):
    ga_kwargs = dict(GA_CONFIG)
    ga_kwargs["seed"] = seed

    if SCENARIO == 2:
        ga_kwargs.update(
            {
                "use_stochastic_evaluation": True,
                "uncertainty_parameters": uncertainty_parameters,
                "n_simulations": FINAL_EVAL_SIMULATIONS,
            }
        )

    ga = build_ga_from_worker_encoding(encoding, **ga_kwargs)
    result = ga.run(**RUN_CONFIG)
    best = result["best"]

    start_times, machines, workers = translate(
        best.sequence,
        best.assignments,
        best.workers,
        encoding.durations(),
    )
    deterministic_makespan = float(makespan(start_times, machines, workers, encoding.durations()))
    worker_balance = float(workload_balance(machines, workers, encoding.durations()))

    raw_function_evaluations = int(result["function_evaluations"])
    competition_function_evaluations = raw_function_evaluations * FINAL_EVAL_SIMULATIONS if SCENARIO == 2 else raw_function_evaluations

    row = {
        "seed": seed,
        "runtime_s": float(result["runtime_s"]),
        "generations": int(result["generations"]),
        "raw_function_evaluations": raw_function_evaluations,
        "function_evaluations": competition_function_evaluations,
        "restarts": int(result["restarts"]),
        "deterministic_makespan": deterministic_makespan,
        "workload_balance": worker_balance,
        "sequence": list(best.sequence),
        "machines": list(best.assignments),
        "workers": list(best.workers),
        "start_times": list(start_times),
    }

    if SCENARIO == 2:
        row["internal_robust_makespan"] = float(best.fitness.get("makespan"))
        row["internal_robust_stdev"] = float(best.fitness.get("robust_makespan_stdev"))
        row["internal_R"] = float(best.fitness.get("R"))
    else:
        row["internal_robust_makespan"] = None
        row["internal_robust_stdev"] = None
        row["internal_R"] = None

    return row


def evaluate_final_solution(row, encoding, uncertainty_parameters=None):
    if SCENARIO == 1:
        row["final_objective"] = row["deterministic_makespan"]
        row["final_robust_makespan"] = None
        row["final_robust_stdev"] = None
        row["final_R"] = None
        row["final_eval_simulations"] = 0
        return row

    end_times = [
        row["start_times"][i] + encoding.durations()[i][int(row["machines"][i])][int(row["workers"][i])]
        for i in range(len(row["start_times"]))
    ]
    results, robust_makespan, robust_makespan_stdev, R = run_n_simulations(
        row["start_times"],
        end_times,
        row["machines"],
        row["workers"],
        encoding.job_sequence(),
        encoding.durations(),
        uncertainty_parameters,
        FINAL_EVAL_SIMULATIONS,
        uncertainty_source=UNCERTAINTY_SOURCE,
        processing_times=True,
    )
    row["final_objective"] = float(robust_makespan)
    row["final_robust_makespan"] = float(robust_makespan)
    row["final_robust_stdev"] = float(robust_makespan_stdev)
    row["final_R"] = float(R)
    row["final_eval_simulations"] = FINAL_EVAL_SIMULATIONS
    row["final_simulation_results"] = [float(x) for x in results]
    return row


def drop_large_columns(df):
    cols = [
        "sequence",
        "machines",
        "workers",
        "start_times",
        "final_simulation_results",
    ]
    keep = [c for c in df.columns if c not in cols]
    return df[keep].copy()

In [5]:
uncertainty_map = load_official_uncertainty_map(OFFICIAL_UNCERTAINTY_FILE)
parser = WorkerBenchmarkParser()

all_rows = []
instance_summaries = []

for instance_name in SELECTED_INSTANCES:
    instance_path = INSTANCES_DIR / instance_name
    encoding = parser.parse_benchmark(str(instance_path))
    uncertainty_parameters, uncertainty_mode = get_uncertainty_parameters(instance_name, encoding, uncertainty_map)

    print(f"Running {instance_name} | scenario={SCENARIO} | uncertainty_mode={uncertainty_mode}")

    instance_rows = []
    for run_idx, seed in enumerate(RUN_SEEDS, start=1):
        print(f"  run {run_idx}/{len(RUN_SEEDS)} seed={seed}")
        row = solve_instance_with_ga(encoding, seed, uncertainty_parameters)
        row["instance"] = instance_name
        row["run"] = run_idx
        row["scenario"] = SCENARIO
        row["uncertainty_mode"] = uncertainty_mode
        row = evaluate_final_solution(row, encoding, uncertainty_parameters)
        instance_rows.append(row)
        all_rows.append(row)

    instance_df = pd.DataFrame(instance_rows)
    metric_col = "final_objective"
    best_idx = instance_df[metric_col].idxmin()
    best_row = instance_df.loc[best_idx]
    instance_summaries.append(
        {
            "instance": instance_name,
            "best_run": int(best_row["run"]),
            "best_seed": int(best_row["seed"]),
            "best_final_objective": float(best_row["final_objective"]),
            "best_deterministic_makespan": float(best_row["deterministic_makespan"]),
            "best_function_evaluations": int(best_row["function_evaluations"]),
            "mean_final_objective": float(instance_df[metric_col].mean()),
            "std_final_objective": float(instance_df[metric_col].std(ddof=0)),
        }
    )

results_df = pd.DataFrame(all_rows)
summary_df = pd.DataFrame(instance_summaries)

print(f"Completed {len(results_df)} runs across {len(summary_df)} instances.")

Running 0_BehnkeGeiger_42_workers.fjs | scenario=2 | uncertainty_mode=demo_fallback
  run 1/10 seed=1000


RecursionError: maximum recursion depth exceeded

In [ ]:
metric_col = "final_objective"
secondary_col = "final_R" if SCENARIO == 2 else "workload_balance"

ranking_df = results_df.copy()
ranking_df["secondary_sort"] = ranking_df[secondary_col].fillna(np.inf)
ranking_df = ranking_df.sort_values(["instance", metric_col, "secondary_sort", "function_evaluations"])
ranking_df["rank_within_instance"] = ranking_df.groupby("instance").cumcount() + 1

instance_rank_scores = ranking_df.groupby("instance", as_index=False)["rank_within_instance"].sum()
overall_rank_score = int(instance_rank_scores["rank_within_instance"].sum())

pd.Series(
    {
        "runs_total": len(results_df),
        "instances_total": results_df["instance"].nunique(),
        "overall_rank_score_demo": overall_rank_score,
        "ranking_metric": metric_col,
        "ranking_secondary": secondary_col,
    }
)

In [ ]:
compact_results_df = drop_large_columns(results_df)
compact_results_df = compact_results_df.sort_values(["instance", "run"]).reset_index(drop=True)
summary_df = summary_df.sort_values("instance").reset_index(drop=True)

compact_results_path = RESULTS_DIR / "run_results.csv"
summary_path = RESULTS_DIR / "instance_summary.csv"
ranking_path = RESULTS_DIR / "ranking_results.csv"
solutions_json_path = RESULTS_DIR / "solutions.json"

compact_results_df.to_csv(compact_results_path, index=False)
summary_df.to_csv(summary_path, index=False)
ranking_df[[
    "instance",
    "run",
    "seed",
    "final_objective",
    "final_R",
    "workload_balance",
    "function_evaluations",
    "rank_within_instance",
]].to_csv(ranking_path, index=False)

solutions_payload = []
for row in results_df.to_dict(orient="records"):
    solutions_payload.append(
        {
            "instance": row["instance"],
            "run": int(row["run"]),
            "seed": int(row["seed"]),
            "final_objective": float(row["final_objective"]),
            "deterministic_makespan": float(row["deterministic_makespan"]),
            "function_evaluations": int(row["function_evaluations"]),
            "sequence": row["sequence"],
            "machines": row["machines"],
            "workers": row["workers"],
            "start_times": row["start_times"],
        }
    )

with solutions_json_path.open("w", encoding="utf-8") as fh:
    json.dump(solutions_payload, fh)

pd.Series(
    {
        "run_results": str(compact_results_path),
        "instance_summary": str(summary_path),
        "ranking_results": str(ranking_path),
        "solutions_json": str(solutions_json_path),
    }
)

In [ ]:
summary_df.head(20)

In [ ]:
plt.figure(figsize=(12, 5))
plot_df = summary_df.copy()
plot_df = plot_df.sort_values("best_final_objective").head(min(15, len(plot_df)))
plt.bar(plot_df["instance"], plot_df["best_final_objective"], color="#2c7fb8")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Best Final Objective")
plt.title("Bestes Resultat je Instanz")
plt.tight_layout()
plt.show()

## Hinweise

- Standardmaessig erzeugt das Notebook fuer Szenario 2 die Unsicherheitsparameter mit `create_uncertainty_vector(...)`.
- Wenn du spaeter offizielle Parameter verwenden willst, setze `OFFICIAL_UNCERTAINTY_FILE` und optional `STRICT_COMPETITION_MODE = True`.
- `run_results.csv` enthaelt die Kennzahlen pro Lauf.
- `solutions.json` enthaelt die eigentlichen Planvektoren fuer eine spaetere Einreichung oder Nachpruefung.
- Die Ranking-Zelle bildet nur einen einfachen notebook-internen Score nach. Fuer eine echte Einreichung sollte das finale Competition-Scoring exakt an das verlangte Format angepasst werden.